### 🧠 What is Query Decomposition?
Query decomposition is the process of taking a complex, multi-part question and breaking it into simpler, atomic sub-questions that can each be retrieved and answered individually.

#### ✅ Why Use Query Decomposition?

- Complex queries often involve multiple concepts

- LLMs or retrievers may miss parts of the original question

- It enables multi-hop reasoning (answering in steps)

- Allows parallelism (especially in multi-agent frameworks)

In [17]:
from langchain.chat_models import init_chat_model
from langchain_core.prompts import PromptTemplate
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.output_parsers import StrOutputParser
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.runnables import RunnableSequence
from langfuse.langchain import CallbackHandler

import os
from dotenv import load_dotenv
load_dotenv()

True

In [18]:
# Step 1: Load and embed the document
loader = TextLoader("langchain_crewai_dataset.txt")
docs = loader.load()

splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50)
chunks = splitter.split_documents(docs)

embedding = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore = FAISS.from_documents(chunks, embedding)
retriever = vectorstore.as_retriever(search_type="mmr", search_kwargs={"k": 4, "lambda_mult": 0.7})
retriever

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 9408.57it/s]


VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x00000280641D7110>, search_type='mmr', search_kwargs={'k': 4, 'lambda_mult': 0.7})

In [19]:
langfuse_trace = CallbackHandler()
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")

# llm=init_chat_model(model="grok-4", model_provider="xai")
# llm=init_chat_model(model="groq:gemma2-9b-it")
llm=init_chat_model(model="groq:openai/gpt-oss-20b")
# llm=init_chat_model(model="qwen3", model_provider="ollama")
# llm=init_chat_model(model="gemma2", model_provider="ollama")

llm

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.4.7', 'langchain': '1.3.9'}}, output_version=None, profile={'name': 'GPT OSS 20B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x0000028063834FC0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000028063835220>, model_name='openai/gpt-oss-20b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [20]:
# Step 3: Query decomposition
decomposition_prompt = PromptTemplate.from_template("""
You are an AI assistant. Decompose the following complex question into 2 to 4 smaller sub-questions for better document retrieval.

Question: "{question}"

Sub-questions:
""")
decomposition_chain = decomposition_prompt | llm | StrOutputParser()

In [21]:
query = "How does LangChain use memory and agents compared to CrewAI?"
decomposition_question=decomposition_chain.invoke({"question": query})


In [22]:
print(decomposition_question)

**Sub-questions:**

1. What memory mechanisms does LangChain provide, and how are they typically configured and used in practice?  
2. How does LangChain define and orchestrate agents, and what are the key components of its agent framework?  
3. What memory strategies does CrewAI employ, and how do they differ from LangChain’s approach?  
4. How does CrewAI construct and manage agents, and how does its agent architecture compare to LangChain’s?


In [23]:
# Step 4: QA chain per sub-question
qa_prompt = PromptTemplate.from_template("""
Use the context below to answer the question.

Context:
{context}

Question: {input}
""")
qa_chain = create_stuff_documents_chain(llm=llm, prompt=qa_prompt)

In [ ]:
# Step 5: Full RAG pipeline logic
def full_query_decomposition_rag_pipeline(user_query):
    # Decompose the query
    sub_qs_text = decomposition_chain.invoke({"question": user_query})
    sub_questions = [q.strip("-•1234567890. ").strip() for q in sub_qs_text.split("\n") if q.strip()]
    
    results = []
    for subq in sub_questions:
        docs = retriever.invoke(subq)
        result = qa_chain.invoke(
            {"input": subq, "context": docs},
            config={"callbacks": [langfuse_trace]}
        )
        results.append(f"Q: {subq}\nA: {result}")
    
    return "\n\n".join(results)

In [25]:
# Step 6: Run
query = "How does LangChain use memory and agents compared to CrewAI?"
final_answer = full_query_decomposition_rag_pipeline(query)
print("✅ Final Answer:\n")
print(final_answer)

sub_qs_text:  Sub-questions:

1. What types of memory (e.g., short‑term, long‑term, context) does LangChain provide, and how are they implemented?
2. How does CrewAI handle memory, and what memory strategies does it offer?
3. In what ways do LangChain and CrewAI differ in their agent design, including agent types, task orchestration, and decision‑making processes?
4. How do these differences in memory and agent architecture affect the practical use cases and performance of each framework?
✅ Final Answer:

Q: Sub-questions:
A: It looks like the sub‑questions you’d like me to answer weren’t included in your message. Could you please list the specific questions you have in mind? Once I have them, I’ll gladly use the context you provided to craft detailed answers.

Q: What types of memory (e.g., short‑term, long‑term, context) does LangChain provide, and how are they implemented?
A: **LangChain Memory Overview**

| Memory type | What it keeps | Typical use‑case | Implementation details |
|